In [6]:
# Import necessary libraries
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from bert_utils import load_data  # Assuming this contains useful functions

In [7]:
# Configuration
MODEL_NAME = "dbmdz/bert-large-cased-finetuned-conll03-english"  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert text into BERT-compatible tokens.
# Load the pre-trained BERT model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()  # Set model to evaluation mode

print("BERT Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERT Model loaded successfully!


In [8]:
# Load the CSV file
CSV_FILE = "speeches_2022-2023.csv"
df = pd.read_csv(CSV_FILE)

# Check the structure of the data
df.head()

,file_id,speaker_name,speaker_party,speech_text,jaar,date,kamer,category,title,document_number,url,meta_url,vergadernummer
0,h-tk-20222023-46-10,De voorzitter,NaN,Ik constateer dat de leden van de fracties van...,2022-2023,2023-01-31,tk,handelingen,10 Stemming motie Ondersteuning Nederlandse de...,NaN,https://zoek.officielebekendmakingen.nl/h-tk-2...,https://zoek.officielebekendmakingen.nl/h-tk-2...,"nr. 46, item 10"
1,h-tk-20222023-68-1,De voorzitter,NaN,Ik open de vergadering van donderdag 30 maart ...,2022-2023,2023-03-30,tk,handelingen,1,NaN,https://zoek.officielebekendmakingen.nl/h-tk-2...,https://zoek.officielebekendmakingen.nl/h-tk-2...,"nr. 68, item 1"
2,h-tk-20222023-18-8,De voorzitter,NaN,Aan de orde is de voortzetting van de begrotin...,2022-2023,2022-11-03,tk,handelingen,8 Begroting Buitenlandse Handel en Ontwikkelin...,NaN,https://zoek.officielebekendmakingen.nl/h-tk-2...,https://zoek.officielebekendmakingen.nl/h-tk-2...,"nr. 18, item 8"
3,h-tk-20222023-18-8,De voorzitter,NaN,Dat gezegd hebbende zou ik graag de minister h...,2022-2023,2022-11-03,tk,handelingen,8 Begroting Buitenlandse Handel en Ontwikkelin...,NaN,https://zoek.officielebekendmakingen.nl/h-tk-2...,https://zoek.officielebekendmakingen.nl/h-tk-2...,"nr. 18, item 8"
4,h-tk-20222023-18-8,Minister Schreinemacher,NaN,"Dank u wel, voorzitter. Dank u wel voor het de...",2022-2023,2022-11-03,tk,handelingen,8 Begroting Buitenlandse Handel en Ontwikkelin...,NaN,https://zoek.officielebekendmakingen.nl/h-tk-2...,https://zoek.officielebekendmakingen.nl/h-tk-2...,"nr. 18, item 8"


In [18]:
# Select a small fraction of data for testing
TEST_FRACTION = 0.005  
df_sample = df.sample(frac=TEST_FRACTION, random_state=42)

print(f" Sampled {len(df_sample)} rows from the dataset.")

 Sampled 444 rows from the dataset.


In [19]:
# Function to split text into 512-token chunks with overlap
def split_into_chunks(text, max_tokens=512, overlap=200):
    """
    Splits a speech into 512-token chunks while ensuring each chunk is within BERT's token limit.
    """
    tokens = tokenizer.encode(text, truncation=False, add_special_tokens=False)  # Tokenize without truncation
    chunks = []
    
    if len(tokens) <= max_tokens:
        return [tokenizer.decode(tokens)]  # Convert back to text if it's within limit
    
    for i in range(0, len(tokens), max_tokens - overlap):
        chunk_tokens = tokens[i : i + max_tokens - 2]  # Reserve space for [CLS] and [SEP]
        
        # Convert tokens back to text
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

    return chunks

print("✅ Splitting function ready!")


✅ Splitting function ready!


In [21]:
from tqdm import tqdm  # Import progress bar module

# Apply the splitting function to the sampled dataframe
df_sample["speech_text_chunks"] = df_sample["speech_text"].apply(split_into_chunks)

# === FUNCTION: TRANSLATE TEXT FROM DUTCH TO ENGLISH (WITH CHUNK HANDLING) ===
def translate_text(text_chunks):
    """Translates a list of text chunks and joins them back together."""
    translated_chunks = []
    
    for chunk in tqdm(text_chunks, desc="Translating chunks", leave=False):  # Progress bar for chunk translation
        try:
            translated_chunk = GoogleTranslator(source="nl", target="en").translate(chunk)
            
            # Ensure translation is a string, otherwise use original text
            if translated_chunk is None:
                translated_chunk = chunk  # Fallback to original text
            
            translated_chunks.append(str(translated_chunk))  # Ensure all chunks are strings
        
        except Exception as e:
            print(f"Translation error: {e}")
            translated_chunks.append(str(chunk))  # Fallback: Keep original text as a string
    
    return " ".join(translated_chunks)  # Combine chunks back into one text

# Apply translation to the segmented speeches with a progress bar
tqdm.pandas(desc="Translating speeches")
df_sample["speech_text_translated"] = df_sample["speech_text_chunks"].progress_apply(translate_text)

print(f"✅ Translated segmented speeches and added new column `speech_text_translated`.")

# Display a few translated rows
df_sample[["speech_text", "speech_text_translated"]].head()


Translating speeches: 100%|███████████████████| 444/444 [07:07<00:00,  1.04it/s]

✅ Translated segmented speeches and added new column `speech_text_translated`.


,speech_text,speech_text_translated
53689,Het is niet alleen ontzettend verdrietig. De v...,It is not only incredibly sad. The question is...
84500,"Ja precies, de staatsschuld. Hij heeft een pun...","Yes exactly, the national debt. He has made a ..."
69448,U schrijft in de brief ook dat beide rechtbank...,You also write in the letter that both courts ...
58960,Zoals ik in de schriftelijke beantwoording heb...,"As I have indicated in the written answer, it ..."
45377,Dat is de motie-Teunissen over de vlaktaks.,That is the motion - Teunissen about the flat ...


In [28]:
# Define the mapping of label numbers to fallacy types
fallacy_label_mapping = {
    1: "Ad Hominem",
    2: "Appeal to Authority",
    3: "Appeal to Emotion",
    4: "False Cause",
    5: "Slippery Slope",
    6: "Slogans"
}

# Function to classify fallacies in a speech
def classify_fallacies(text):
    """Runs BERT model on a given speech text and returns detected fallacies with proper word reconstruction."""
    
    if not isinstance(text, str) or text.strip() == "":
        return []  # Handle empty input

    chunks = split_into_chunks(text)  # Split long speeches
    predictions = []

    for chunk in chunks:
        try:
            inputs = tokenizer(
                chunk,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            )
            inputs = {key: val.to(device) for key, val in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)

            logits = outputs.logits
            predicted_labels = torch.argmax(logits, dim=2)  # Get highest probability label

            tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

            # === REBUILD WORDS FROM SUBWORDS ===
            fallacies = []
            current_word = ""
            current_label = None

            for token, label in zip(tokens, predicted_labels[0]):
                word_part = token.replace("##", "")  # Remove subword markers
                
                if token.startswith("##"):  
                    # If it's a continuation subword, append to the current word
                    current_word += word_part
                else:
                    # If it's a new word, store the previous word (if it had a fallacy label)
                    if current_word and current_label is not None:
                        fallacies.append((current_word, fallacy_label_mapping.get(current_label, "Unknown")))

                    # Start a new word
                    current_word = word_part
                    current_label = label.item() if label.item() != 0 else None  # Ignore "O" labels

            # Add last word if it had a label
            if current_word and current_label is not None:
                fallacies.append((current_word, fallacy_label_mapping.get(current_label, "Unknown")))

            predictions.extend(fallacies)

        except Exception as e:
            print(f"⚠️ Error processing chunk: {e}")
            continue  # Skip problematic chunk but continue processing

    return predictions


In [29]:
import nltk
from tqdm import tqdm

nltk.download('punkt')  # Ensure sentence tokenizer is available

# Function to detect fallacies in each sentence separately
def detect_fallacies_in_speech(speech):
    """Splits speech into sentences, detects fallacies, and returns structured output."""
    
    sentences = nltk.sent_tokenize(speech)  # Split speech into sentences
    sentence_fallacies = []  # Store fallacies per sentence
    
    for sentence in sentences:
        fallacies = classify_fallacies(sentence)  # Detect fallacies in this sentence
        
        if fallacies:  # If fallacies are found, format them
            formatted_fallacies = ", ".join([f"{word} → {fallacy}" for word, fallacy in fallacies])
            sentence_fallacies.append(f"{sentence} → {formatted_fallacies}")  # Attach fallacies to the sentence
    
    return " | ".join(sentence_fallacies) if sentence_fallacies else "No fallacy detected"

print("✅ `detect_fallacies_in_speech` function loaded successfully!")


✅ `detect_fallacies_in_speech` function loaded successfully!


[nltk_data] Downloading package punkt to /Users/markus/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [30]:
from tqdm import tqdm  # Ensure tqdm is imported for the progress bar

TEXT_COLUMN = "speech_text_translated"  

# Apply fallacy detection with sentence-level tracking
tqdm.pandas(desc="Detecting fallacies per sentence")
df_sample["fallacy_predictions"] = df_sample[TEXT_COLUMN].progress_apply(detect_fallacies_in_speech)

print("✅ Fallacy detection applied with sentence tracking!")

# Display a few rows with detected fallacies
df_sample[["speech_text_translated", "fallacy_predictions"]].head()


Detecting fallacies per sentence: 100%|███████| 444/444 [28:43<00:00,  3.88s/it]

✅ Fallacy detection applied with sentence tracking!


,speech_text_translated,fallacy_predictions
53689,It is not only incredibly sad. The question is...,The question is of course what consequences th...
84500,"Yes exactly, the national debt. He has made a ...",If you are now going to look at Europe and if ...
69448,You also write in the letter that both courts ...,No fallacy detected
58960,"As I have indicated in the written answer, it ...",I agree with Mrs. Paulusma. → Paulusma → False...
45377,That is the motion - Teunissen about the flat ...,That is the motion - Teunissen about the flat ...


In [31]:
# Define the output file path
OUTPUT_FILE = "fallacy_detection_results.csv"

# Save the dataframe with fallacy predictions
df_sample.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Fallacy detection results saved to {OUTPUT_FILE}!")


✅ Fallacy detection results saved to fallacy_detection_results.csv!
